In [ ]:
from langchain.agents import create_agent
from langchain_ollama import ChatOllama

# 初始化Ollama平台的模型
model = ChatOllama(
    model="qwen3.5:4b",
    temperature=1.0,
)

# =================测试====================
# 初始化Agent，传入初始化好的model
agent = create_agent(model=model, tools=[])

response = agent.invoke({"messages": [{"role": "user", "content": "你是谁？"}]})
print(response)

In [2]:
import os

from dotenv import load_dotenv
from langchain.chat_models import init_chat_model
from langchain_core.tools import tool
from langchain_deepseek import ChatDeepSeek
from langchain_ollama import ChatOllama
from langchain_openai import ChatOpenAI, OpenAI
from pydantic import SecretStr
from rich import print as rich_print

load_dotenv(override=True)  # 加载环境变量（如果有本地 .env 配置）


def run_native_sdk():
    """底层原生 SDK 调用"""

    api_key_str = os.getenv("DEEPSEEK_API_KEY")

    # 初始化原生客户端（可配置 base_url 对接 DeepSeek、硅基流动或 OpenAI）
    client = OpenAI(api_key=api_key_str, base_url="https://api.deepseek.com/v1")

    response = client.chat.completions.create(
        model="deepseek-chat",
        messages=[{"role": "user", "content": "一句话介绍你自己？"}],
        reasoning_effort="high",
        extra_body={"thinking": {"type": "disabled"}},  # enabled / disabled
        temperature=0.1,
    )
    # rich_print(response.choices[0].message.reasoning_content)
    rich_print(response.choices[0].message.content)
    rich_print(response)


def run_ollama_llm():
    """运行 Ollama 本地模型"""

    print("--- 正在测试 Ollama 本地模型 ---")
    # 【方式一】init_chat_model
    # 注意：init_chat_model 会自动解析 "ollama:模型名称" 并寻找对应的集成包 (langchain-ollama)
    llm = init_chat_model("ollama:deepseek-r1:14b")  # uv add langchain-ollama
    # llm = init_chat_model("deepseek-r1:14b", model_provider="ollama")
    # llm = init_chat_model(model="deepseek-r1:14b", model_provider="ollama")
    # 【方式二】显式使用 ChatOllama 类
    llm = ChatOllama(
        model="deepseek-r1:14b",
        temperature=0.7,
        # 可传入其他专属参数
    )
    # 【方式三】OpenAI 兼容模式
    # 接入 DeepSeek 官方 API 或兼容 OpenAI 格式的第三方服务
    api_key_str = os.getenv("DEEPSEEK_API_KEY")
    api_key = SecretStr(api_key_str) if api_key_str else None
    llm = ChatOpenAI(
        model="deepseek-chat",
        base_url="https://api.deepseek.com/v1",  # 替换为你的目标 API 地址
        api_key=api_key,  # 或通过环境变量 DEEPSEEK_API_KEY 自动读取
        temperature=0.1,
    )
    response = llm.invoke([{"role": "user", "content": "一句话介绍你自己？"}])
    # 打印返回结果（Ollama 的 DeepSeek-R1 模型通常会输出思考过程 <think> 和最终回答）
    rich_print(response.content)


def run_online_deepseek_llm():
    """运行在线 DeepSeek 官方模型"""

    print("--- 正在测试在线 DeepSeek 模型 ---")
    api_key_str = os.getenv("DEEPSEEK_API_KEY")
    api_key = SecretStr(api_key_str) if api_key_str else None
    # 【方式一】使用 init_chat_model 统一工厂函数（通过 OpenAI 兼容模式对接 DeepSeek 官方）
    # 需要安装依赖: uv add langchain-openai
    # llm = init_chat_model(
    #     model="deepseek-chat",
    #     model_provider="openai",
    #     base_url="https://api.deepseek.com/v1",
    #     api_key=api_key_str
    # )
    llm = init_chat_model(
        "openai:deepseek-chat", base_url="https://api.deepseek.com/v1", api_key=api_key_str
    )
    # 【方式二】使用 LangChain 官方专属的 ChatDeepSeek 类运行在线模型
    # 不需要手动指定 base_url，它会自动连接到 DeepSeek 官方接口
    llm = ChatDeepSeek(
        model="deepseek-chat",
        # api_key=api_key, # 也可以不写，会自动读取环境变量 DEEPSEEK_API_KEY
    )
    # 【方式三】显式使用 ChatOpenAI 类对接 DeepSeek 官方 API（最常用、对 V3/R1 支持最完整）
    llm = ChatOpenAI(
        model="deepseek-reasoner",  # 对应 DeepSeek-R1 深度思考推理模型
        base_url="https://api.deepseek.com/v1",  # 官方 API 端点
        api_key=api_key,
    )
    response = llm.invoke([{"role": "user", "content": "一句话介绍你自己？"}])
    # 打印返回结果（如果是 deepseek-reasoner，还可以通过 response.additional_kwargs 获取思考过程）
    rich_print(response.content)


def run_siliconflow_llm():
    """三方聚合平台（以硅基流动 SiliconFlow 为例）"""
    print("--- 正在测试硅基流动托管模型 ---")

    api_key_str = os.getenv("SILICONFLOW_API_KEY")
    api_key = SecretStr(api_key_str) if api_key_str else None

    # 【方式一】通过 init_chat_model 结合 OpenAI 兼容参数初始化硅基流动模型
    # llm = init_chat_model(
    #     model="deepseek-ai/DeepSeek-V3",  # 硅基流动上的模型路径
    #     model_provider="openai",          # 声明基于 openai 协议
    #     base_url="https://api.siliconflow.cn/v1",
    #     api_key=api_key_str
    # )
    llm = init_chat_model(
        model="openai:deepseek-ai/DeepSeek-V3",  # 硅基流动上的模型路径
        base_url="https://api.siliconflow.cn/v1",
        api_key=api_key_str,
    )

    # 【方式二】硅基流动完美兼容 OpenAI 格式，直接使用 ChatOpenAI 并替换 base_url 即可
    llm = ChatOpenAI(
        # model="deepseek-ai/DeepSeek-V3",  # 硅基流动上的模型路径标识
        model="Qwen/Qwen2.5-7B-Instruct",
        base_url="https://api.siliconflow.cn/v1",
        api_key=api_key,
    )
    response = llm.invoke([{"role": "user", "content": "一句话介绍你自己？"}])
    rich_print(response.content)


# 1. 定义一个工具
@tool
def get_weather(location: str) -> str:
    """查询指定城市的天气情况。"""
    return f"{location}今天天气晴朗，气温 22.2°C。"


def run_agent_workflow():
    """层 3：智能体与工作流层调用"""

    # 2. 创建带有工具的 Agent
    # agent = create_agent(
    #     model="deepseek-chat",
    #     tools=[get_weather]
    # )

    api_key_str = os.getenv("SILICONFLOW_API_KEY")
    api_key = SecretStr(api_key_str) if api_key_str else None
    llm = ChatOpenAI(
        # model="deepseek-ai/DeepSeek-V3",  # 硅基流动上的模型路径标识
        model="Qwen/Qwen2.5-7B-Instruct",
        base_url="https://api.siliconflow.cn/v1",
        api_key=api_key,
    )
    agent = create_agent(
        model=llm,
        tools=[get_weather],
    )

    # 3. 异步执行调用（Agent 会自主决定是否调用工具）
    response = agent.invoke({"messages": [("user", "杭州今天天气怎么样？")]})

    # 打印最终回答
    print("【Agent 智能体响应】:", response["messages"][-1].content)


# run_native_sdk()
# run_ollama_llm()
# run_online_deepseek_llm()
# run_siliconflow_llm()
run_agent_workflow()

【Agent 智能体响应】: 杭州今天天气晴朗，气温为 22.2°C。
